# 第 18 天：Alpha101复现2

> 所属阶段：从“经典单因子”进入“公式化 Alpha 工厂”
> 今日主题：Alpha101复现2
> 必做：复现6-10号
> 选做：扩展改造
> 目标产出：Notebook

今天开始，你会从“我知道 PE、ROE、动量这些传统因子”切换到另一个世界：公式化 Alpha。

传统因子像是几种常见食材：价值、质量、动量、波动率、流动性。  
Alpha101 更像是一套厨房语言：`rank`、`delay`、`delta`、`correlation`、`ts_rank`、`adv20`、`vwap` 这些算子可以拼出很多味道不同的信号。

这一阶段最重要的不是背公式，而是学会三件事：

1. 看懂公式里的每个算子在说什么。
2. 把公式翻译成稳定、可检查、无未来函数的 Python。
3. 用因子研究流程判断它有没有研究价值，而不是只看公式是否“高级”。



## 0. 今天你要真正学会什么？

今天的任务是复现 Alpha 006, 007, 008, 009, 010。

注意，所谓“复现”不是把一行公式打进 Python 就结束。你要完成一整条研究链路：

1. 拆公式：字段、算子、窗口、方向。
2. 写代码：每个中间变量都能检查。
3. 产出因子矩阵：日期 x 股票。
4. 做 IC 检验：看它和未来收益有没有关系。
5. 做横向比较：这 5 个 Alpha 是否高度重复。
6. 做扩展改造：尝试行业中性化、窗口变化、持有期变化。

如果你能把今天的 5 个公式讲清楚，你就不再只是“复制 Alpha101”，而是在建立自己的 Alpha 工厂。

## 1. 先建立直觉：公式化 Alpha 的四种味道

今天这组公式大多围绕四个问题：

### 1.1 价格在短期窗口里是否极端？

这类信号关心“今天是否已经走到一个很偏的位置”。  
极端本身不直接等于买入或卖出，但它提醒你市场行为可能进入非正常状态。

### 1.2 成交量是否配合价格？

只看价格，容易误判。  
同样的涨幅，如果成交量很低，可能只是轻微波动；如果成交量很高，可能代表资金集中参与。

### 1.3 开盘、收盘、VWAP 的相对位置说明什么？

开盘价像市场开场投票，收盘价像当天最终表态，VWAP 更接近平均成交成本。  
三者之间的距离，往往包含日内交易结构。

### 1.4 短周期信号为什么要特别关注换手？

Alpha101 很多公式变化很快。  
信号今天强，明天可能就弱。  
如果换手太高，即使 IC 有一点优势，也可能被交易成本吃掉。

## 2. 今日公式地图

| Alpha | 直觉关键词 | 字段 | 算子 |
|---|---|---|---|
| 006 | 开盘价与成交量的短期相关 | open、volume | correlation |
| 007 | 异常成交量条件下的价格变化方向 | close、volume、adv20 | delta、abs、ts_rank、sign |
| 008 | 开盘价与收益乘积的延迟差 | open、returns | sum、delay、rank |
| 009 | 短期价格变化的条件反转 | close | delta、ts_min、ts_max |
| 010 | 条件反转后的横截面排名 | close | delta、ts_min、ts_max、rank |

## 3. 复现前的数据准备

为了让你单独打开这一天也能运行，下面重新构造一份模拟行情数据。  
真实研究时，请把这部分替换成你的真实日频数据。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(202620)

dates = pd.bdate_range("2022-01-03", periods=360)
assets = [f"S{i:03d}" for i in range(1, 51)]

industries = pd.Series(
    np.random.choice(["消费", "科技", "制造", "医药", "金融"], size=len(assets)),
    index=assets,
    name="industry",
)
size_score = pd.Series(np.random.normal(0, 1, len(assets)), index=assets, name="size_score")
asset_beta = pd.Series(np.random.uniform(0.75, 1.35, len(assets)), index=assets, name="beta")
asset_drift = pd.Series(np.random.normal(0.00015, 0.00018, len(assets)), index=assets, name="drift")

market_return = np.random.normal(0.00025, 0.010, len(dates))
style_shock = np.random.normal(0, 0.004, (len(dates), len(assets)))
idiosyncratic = np.random.normal(0, 0.018, (len(dates), len(assets)))

daily_return = (
    market_return[:, None] * asset_beta.values[None, :]
    + asset_drift.values[None, :]
    + style_shock
    + idiosyncratic
)

close = pd.DataFrame(
    30 * np.exp(np.cumsum(daily_return, axis=0)),
    index=dates,
    columns=assets,
)

overnight = pd.DataFrame(
    np.random.normal(0, 0.006, close.shape),
    index=dates,
    columns=assets,
)
open_ = close.shift(1) * (1 + overnight)
open_.iloc[0] = close.iloc[0] * (1 + overnight.iloc[0])

intraday_span = pd.DataFrame(
    np.random.uniform(0.002, 0.035, close.shape),
    index=dates,
    columns=assets,
)
high = pd.DataFrame(
    np.maximum(open_.to_numpy(), close.to_numpy()) * (1 + intraday_span.to_numpy()),
    index=dates,
    columns=assets,
)
low = pd.DataFrame(
    np.minimum(open_.to_numpy(), close.to_numpy()) * (1 - intraday_span.to_numpy()),
    index=dates,
    columns=assets,
)

base_volume = (900_000 * np.exp(size_score.values))[None, :]
volume = pd.DataFrame(
    base_volume
    * np.random.lognormal(mean=0, sigma=0.45, size=close.shape)
    * (1 + close.pct_change().fillna(0).abs().to_numpy() * 12),
    index=dates,
    columns=assets,
)
vwap = (open_ + high + low + close) / 4
returns = close.pct_change()
adv20 = volume.rolling(20).mean()
future_5d = close.shift(-5) / close - 1

print("数据形状：")
print({
    "close": close.shape,
    "open": open_.shape,
    "high": high.shape,
    "low": low.shape,
    "volume": volume.shape,
    "future_5d": future_5d.shape,
})
print("\n行业分布：")
print(industries.value_counts())


## 4. 复现前的算子准备

这套算子沿用第 16 天的最小 Alpha101 算子库。  
它的设计原则是：输入输出都保持日期 x 股票矩阵，便于组合和批量测试。


In [ ]:
def cs_rank(df: pd.DataFrame) -> pd.DataFrame:
    """横截面排名：每天在所有股票之间排序，输出 0 到 1 附近的百分位。"""
    return df.rank(axis=1, pct=True)


def delay(df: pd.DataFrame, n: int = 1) -> pd.DataFrame:
    """向后取 n 期，避免把今天之后的信息放进今天。"""
    return df.shift(n)


def delta(df: pd.DataFrame, n: int = 1) -> pd.DataFrame:
    """当前值减去 n 期前的值。"""
    return df - delay(df, n)


def ts_mean(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).mean()


def ts_sum(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).sum()


def ts_std(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).std()


def ts_min(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).min()


def ts_max(df: pd.DataFrame, window: int) -> pd.DataFrame:
    return df.rolling(window).max()


def ts_rank(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """时间序列排名：今天的值在过去 window 天里排第几。"""
    return df.rolling(window).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1],
        raw=False,
    )


def ts_argmax(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """过去 window 天最大值出现的位置，1 表示窗口第一天，window 表示今天。"""
    return df.rolling(window).apply(lambda x: np.argmax(x) + 1, raw=True)


def corr(a: pd.DataFrame, b: pd.DataFrame, window: int) -> pd.DataFrame:
    return a.rolling(window).corr(b)


def cov(a: pd.DataFrame, b: pd.DataFrame, window: int) -> pd.DataFrame:
    return a.rolling(window).cov(b)


def signed_power(df: pd.DataFrame, power: float) -> pd.DataFrame:
    return np.sign(df) * (df.abs() ** power)


def decay_linear(df: pd.DataFrame, window: int) -> pd.DataFrame:
    weights = np.arange(1, window + 1, dtype=float)
    weights = weights / weights.sum()
    return df.rolling(window).apply(lambda x: np.dot(x, weights), raw=True)


def safe_clean(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan)


def neutralize_by_group(factor: pd.DataFrame, groups: pd.Series) -> pd.DataFrame:
    """简化版行业中性化：每个交易日、每个行业内减去行业均值。"""
    result = pd.DataFrame(index=factor.index, columns=factor.columns, dtype=float)
    for _, cols in groups.groupby(groups).groups.items():
        cols = list(cols)
        block = factor[cols]
        result[cols] = block.sub(block.mean(axis=1), axis=0)
    return result


def calc_rank_ic(factor: pd.DataFrame, label: pd.DataFrame) -> pd.Series:
    """按日期计算 Rank IC。"""
    factor = safe_clean(factor)
    label = label.reindex_like(factor)
    return factor.rank(axis=1).corrwith(label.rank(axis=1), axis=1)


def factor_report(factor: pd.DataFrame, label: pd.DataFrame, name: str) -> pd.Series:
    """给单个因子生成一个轻量研究摘要。"""
    f = safe_clean(factor)
    ic = calc_rank_ic(f, label).dropna()
    q = f.rank(axis=1, pct=True)
    long_leg = label.where(q >= 0.8).mean(axis=1)
    short_leg = label.where(q <= 0.2).mean(axis=1)
    ls = (long_leg - short_leg).dropna()
    turnover = q.ge(0.8).astype(float).diff().abs().mean(axis=1).dropna()

    return pd.Series({
        "factor": name,
        "ic_mean": ic.mean(),
        "ic_ir": ic.mean() / ic.std() if ic.std() != 0 else np.nan,
        "ic_positive_ratio": (ic > 0).mean(),
        "long_short_mean": ls.mean(),
        "long_short_win_rate": (ls > 0).mean(),
        "top_bucket_turnover": turnover.mean(),
        "valid_days": len(ic),
    })


def describe_factor_set(factors: dict[str, pd.DataFrame], label: pd.DataFrame) -> pd.DataFrame:
    rows = [factor_report(factor, label, name) for name, factor in factors.items()]
    return pd.DataFrame(rows).set_index("factor").sort_values("ic_mean", ascending=False)


print("核心算子已准备好：cs_rank、delay、delta、ts_rank、corr、cov、neutralize、IC report")


## 5. 逐个拆解今天的 5 个 Alpha


### Alpha 006：开盘价与成交量的短期相关

**一句话直觉**：价格水平和成交活跃度同步时，可能代表被资金推动后的拥挤。

**用到的数据字段**：open、volume

**核心算子**：correlation

**翻译时要注意**：

- 先确认这个公式是横截面逻辑、时间序列逻辑，还是两者混合。
- 涉及 rolling 窗口时，前几天自然会出现缺失值。
- 最后输出的因子方向不一定天然是“越大越好”，必须用 IC 来判断。
- 不要只因为公式复杂就默认它有效。

**研究笔记写法**：


Alpha 006
直觉：
字段：
算子：
可能捕捉的行为：
风险暴露：
初步 IC：
是否值得继续：


### Alpha 007：异常成交量条件下的价格变化方向

**一句话直觉**：当成交量高于 20 日均量时，才启用价格变化强弱信号。

**用到的数据字段**：close、volume、adv20

**核心算子**：delta、abs、ts_rank、sign

**翻译时要注意**：

- 先确认这个公式是横截面逻辑、时间序列逻辑，还是两者混合。
- 涉及 rolling 窗口时，前几天自然会出现缺失值。
- 最后输出的因子方向不一定天然是“越大越好”，必须用 IC 来判断。
- 不要只因为公式复杂就默认它有效。

**研究笔记写法**：


Alpha 007
直觉：
字段：
算子：
可能捕捉的行为：
风险暴露：
初步 IC：
是否值得继续：


### Alpha 008：开盘价与收益乘积的延迟差

**一句话直觉**：观察短期价格状态和收益状态的组合是否发生变化。

**用到的数据字段**：open、returns

**核心算子**：sum、delay、rank

**翻译时要注意**：

- 先确认这个公式是横截面逻辑、时间序列逻辑，还是两者混合。
- 涉及 rolling 窗口时，前几天自然会出现缺失值。
- 最后输出的因子方向不一定天然是“越大越好”，必须用 IC 来判断。
- 不要只因为公式复杂就默认它有效。

**研究笔记写法**：


Alpha 008
直觉：
字段：
算子：
可能捕捉的行为：
风险暴露：
初步 IC：
是否值得继续：


### Alpha 009：短期价格变化的条件反转

**一句话直觉**：如果近期价格变化持续同向，就顺势；否则取反。

**用到的数据字段**：close

**核心算子**：delta、ts_min、ts_max

**翻译时要注意**：

- 先确认这个公式是横截面逻辑、时间序列逻辑，还是两者混合。
- 涉及 rolling 窗口时，前几天自然会出现缺失值。
- 最后输出的因子方向不一定天然是“越大越好”，必须用 IC 来判断。
- 不要只因为公式复杂就默认它有效。

**研究笔记写法**：


Alpha 009
直觉：
字段：
算子：
可能捕捉的行为：
风险暴露：
初步 IC：
是否值得继续：


### Alpha 010：条件反转后的横截面排名

**一句话直觉**：在 009 的基础上加横截面排序，让信号更适合选股。

**用到的数据字段**：close

**核心算子**：delta、ts_min、ts_max、rank

**翻译时要注意**：

- 先确认这个公式是横截面逻辑、时间序列逻辑，还是两者混合。
- 涉及 rolling 窗口时，前几天自然会出现缺失值。
- 最后输出的因子方向不一定天然是“越大越好”，必须用 IC 来判断。
- 不要只因为公式复杂就默认它有效。

**研究笔记写法**：


Alpha 010
直觉：
字段：
算子：
可能捕捉的行为：
风险暴露：
初步 IC：
是否值得继续：


## 6. Python 复现：从公式到因子矩阵

下面的代码会一次性生成今天的 5 个 Alpha。

为了方便调试，我建议你在真实研究中保留中间变量，比如 `x007`、`spread`、`d1`。  
中间变量不是啰嗦，它们是你排查公式方向和数据异常的抓手。


In [ ]:
alpha006 = -1 * corr(open_, volume, 10)

x007 = -1 * ts_rank(delta(close, 7).abs(), 60) * np.sign(delta(close, 7))
alpha007 = x007.where(volume > adv20, -1.0)

x008 = ts_sum(open_, 5) * ts_sum(returns, 5)
alpha008 = -1 * cs_rank(x008 - delay(x008, 10))

d1 = delta(close, 1)
alpha009 = d1.where(ts_min(d1, 5) > 0, d1.where(ts_max(d1, 5) < 0, -1 * d1))

d1 = delta(close, 1)
alpha010 = cs_rank(d1.where(ts_min(d1, 4) > 0, d1.where(ts_max(d1, 4) < 0, -1 * d1)))

alphas = {
    "alpha006": alpha006,
    "alpha007": alpha007,
    "alpha008": alpha008,
    "alpha009": alpha009,
    "alpha010": alpha010,
}

alpha_snapshot = pd.concat(
    {name: factor.iloc[-1] for name, factor in alphas.items()},
    axis=1,
)

print("最后一个交易日的 Alpha 截面快照：")
print(alpha_snapshot.describe().round(4))


## 7. 单因子检查：先拿 Alpha 006 做显微镜


In [ ]:
target_name = "alpha006"
target_factor = alphas[target_name]

print(f"{target_name} 因子摘要：")
print(target_factor.stack().describe().round(4))

target_ic = calc_rank_ic(target_factor, future_5d).dropna()
print("\nRank IC 摘要：")
print(target_ic.describe().round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_ic.cumsum().plot(ax=axes[0], title=f"{target_name} 累计 Rank IC")

target_quantile = target_factor.rank(axis=1, pct=True)
target_ls = (
    future_5d.where(target_quantile >= 0.8).mean(axis=1)
    - future_5d.where(target_quantile <= 0.2).mean(axis=1)
).dropna()
target_ls.cumsum().plot(ax=axes[1], title=f"{target_name} 多空累计收益")

plt.tight_layout()
plt.show()
plt.close()


## 8. 批量评价：5 个 Alpha 放在同一张表里

单个公式看起来好不代表它真的好。  
你要把它和同组公式放在一起比较。


In [ ]:
summary = describe_factor_set(alphas, future_5d)
cols = ["ic_mean", "ic_ir", "ic_positive_ratio", "long_short_mean", "long_short_win_rate", "top_bucket_turnover", "valid_days"]
print(summary[cols].round(4))


In [ ]:
summary[["ic_mean", "ic_ir", "top_bucket_turnover"]].plot(
    kind="bar",
    figsize=(10, 4),
    title="今日 5 个 Alpha 的 IC 与换手对比",
)
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()
plt.close()


## 9. 相关性检查：它们是不是在重复表达同一件事？

如果 5 个 Alpha 彼此高度相关，你不一定需要都保留。  
因子库追求的是“有效且不完全重复”。


In [ ]:
factor_panel = pd.concat(
    {name: factor.stack() for name, factor in alphas.items()},
    axis=1,
)
factor_corr = factor_panel.corr()
print(factor_corr.round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(factor_corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(factor_corr.columns)))
ax.set_xticklabels(factor_corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(factor_corr.index)))
ax.set_yticklabels(factor_corr.index)
ax.set_title("Alpha 相关性矩阵")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()
plt.close()


## 10. 行业中性化：去掉一层可能的伪有效

公式化 Alpha 可能无意中选中了某些行业。  
今天先做一个简化版行业中性化，观察结果是否明显变化。


In [ ]:
neutralized_alphas = {
    name: neutralize_by_group(factor, industries)
    for name, factor in alphas.items()
}

raw_summary = describe_factor_set(alphas, future_5d)
neutral_summary = describe_factor_set(neutralized_alphas, future_5d)

neutral_compare = pd.DataFrame({
    "raw_ic": raw_summary["ic_mean"],
    "industry_neutral_ic": neutral_summary["ic_mean"],
    "raw_turnover": raw_summary["top_bucket_turnover"],
    "industry_neutral_turnover": neutral_summary["top_bucket_turnover"],
})

print(neutral_compare.round(4))


## 11. 持有期敏感性：同一个 Alpha 对不同标签是否稳定？

一个短周期 Alpha 可能只对 1 日或 5 日有效，对 20 日没有意义。  
所以不要只盯一个持有期。


In [ ]:
holding_rows = []
for holding_days in [1, 5, 10, 20]:
    label = close.shift(-holding_days) / close - 1
    for name, factor in alphas.items():
        ic = calc_rank_ic(factor, label).dropna()
        holding_rows.append({
            "holding_days": holding_days,
            "factor": name,
            "ic_mean": ic.mean(),
            "ic_ir": ic.mean() / ic.std() if ic.std() != 0 else np.nan,
            "positive_ratio": (ic > 0).mean(),
        })

holding_report = pd.DataFrame(holding_rows)
print(holding_report.pivot(index="factor", columns="holding_days", values="ic_mean").round(4))


## 12. 扩展改造：从复现到研究

复现只是起点。你可以从三个方向改造今天的 Alpha：

### 12.1 改窗口

把窗口从 3 改成 5，从 10 改成 20，观察 IC 和换手是否更平稳。

### 12.2 改字段

例如把 `close` 换成 `vwap`，把 `volume` 换成 `volume / adv20`。  
这不是随便乱改，而是在问：相对成交活跃度是否比绝对成交量更稳健？

### 12.3 改预处理

先做行业中性化、去极值、标准化，再看结果是否改善。


In [ ]:
standardized_alphas = {}
for name, factor in neutralized_alphas.items():
    clipped = factor.clip(
        lower=factor.quantile(0.01, axis=1),
        upper=factor.quantile(0.99, axis=1),
        axis=0,
    )
    standardized_alphas[name] = clipped.sub(clipped.mean(axis=1), axis=0).div(clipped.std(axis=1), axis=0)

std_summary = describe_factor_set(standardized_alphas, future_5d)
print(std_summary[["ic_mean", "ic_ir", "top_bucket_turnover"]].round(4))


## 13. 今日研究日志模板


日期：
复现 Alpha：

Alpha 006：
直觉：
代码是否通过：
IC：
换手：
保留/观察/丢弃：

Alpha 007：
直觉：
代码是否通过：
IC：
换手：
保留/观察/丢弃：

Alpha 008：
直觉：
代码是否通过：
IC：
换手：
保留/观察/丢弃：

Alpha 009：
直觉：
代码是否通过：
IC：
换手：
保留/观察/丢弃：

Alpha 010：
直觉：
代码是否通过：
IC：
换手：
保留/观察/丢弃：

今天最值得继续研究的公式：
原因：

今天最可能是噪声的公式：
原因：

下一步改造：


## 14. 常见坑深挖

### 坑 1：把复现等同于复制

复制公式只能得到一列数字。  
复现意味着你知道每个中间变量为什么这样算。

### 坑 2：不检查方向

有些 Alpha 原始方向可能是越大越差。  
如果你不看 IC 正负，只看绝对值，就可能把多头和空头做反。

### 坑 3：没有处理无穷大

成交量、价格差、比例变量都可能产生极端值。  
公式化 Alpha 的第一步永远是 `replace inf -> NaN`。

### 坑 4：只看 IC 均值

IC 均值为正但波动极大，可能不稳定。  
ICIR 和正 IC 比例同样重要。

### 坑 5：忽略相关性

两个 Alpha 都有效，但相关性 0.95，加入组合后边际贡献可能很低。

### 坑 6：把行业暴露当 Alpha

行业中性化前有效，中性化后消失，说明你要重新解释它。

### 坑 7：忘记写失败结果

失败公式也有价值。  
它能告诉你哪些结构在当前市场或当前数据里不稳。

## 15. 今日作业

### 作业 A：逐个解释 5 个公式

不要写数学符号，改用自然语言解释每个 Alpha。

### 作业 B：完成批量评价表

记录每个 Alpha 的：

- IC 均值
- ICIR
- 正 IC 比例
- 多空收益
- 换手率

### 作业 C：做一次持有期敏感性分析

比较 1、5、10、20 日标签下的 IC。

### 作业 D：选一个 Alpha 做改造

改一个窗口或字段，并解释为什么这么改。

### 作业 E：写入 Alpha Zoo

把值得保留的公式放入你的 Alpha Zoo：


类别：
公式：
直觉：
数据字段：
窗口：
IC：
风险：
是否保留：


## 16. 面试式自测

### 问 1：为什么 Alpha101 复现时要保留中间变量？

答案：因为中间变量能帮助你检查公式方向、缺失值、窗口逻辑和异常值。

### 问 2：IC 均值高就一定能交易吗？

答案：不一定。还要看换手、成本、稳定性、相关性、容量和样本外表现。

### 问 3：为什么要做持有期敏感性？

答案：因为不同 Alpha 的作用周期不同，短周期信号不一定能预测长期收益。

### 问 4：如果行业中性化后 IC 消失，你怎么解释？

答案：原始效果可能主要来自行业暴露，而不是公式本身。

### 问 5：相关性矩阵有什么用？

答案：判断这些 Alpha 是否重复表达同一种市场行为，帮助后续去冗余。

## 17. 今日复盘模板


今天最清楚的公式：

今天最绕的公式：

我发现的一个代码风险：

我发现的一个经济直觉：

我决定保留的 Alpha：

我决定暂时观察的 Alpha：

我决定丢弃的 Alpha：

下一节课前要补的内容：


## 18. 明天预告：Alpha101 复现 3：11-15 号

下一天会继续复现下一组 Alpha。  
你会逐渐看到一个模式：很多公式表面不同，本质都在表达价格、成交量、相对位置和短期变化之间的关系。

## 19. 一句话收尾

真正的 Alpha101 学习，不是看懂 101 个公式，而是练出“任何公式都能拆、能写、能验、能解释”的研究肌肉。

## 20. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须进行严格的样本外测试和交易成本评估。
